# KV cache, measured

Companion notebook to **Lecture 27/28 — Attention and the KV cache**, §5.1–5.2. Qwen3-0.6B generates text with and without its cache, then the cache is read back: tensor shapes, total bytes, and the formula they should match.

- **Model:** `Qwen/Qwen3-0.6B` in BF16, about 1.2 GB. No login needed.
- **Runtime:** the free CPU runtime is enough. Part 1 takes a minute or two; part 2 takes seconds.

![](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*dSCIzMEXEdNAbPtUXjIwaQ.png)  
Image credits: [Medium: The Rise of LLMs: From GPT to Modern Innovations](https://medium.com/@lmpo/decoder-only-transformer-models-a-comprehensive-overview-c54dc3286d71)

In [1]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16)
model.eval()
cfg = model.config

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

## 1. With and without the cache · §5.1

Without a cache, every new token reruns the **whole sequence** and recomputes the keys and values of every past token. With one, the model is fed only the newest token and reads past keys and values from the cache.

**Predict before you run:**

1. Do the two methods generate the same tokens?
2. As the context grows from 64 to 512 tokens, which per-token time grows, and which stays roughly flat?

In [3]:
import statistics


def generate_without_cache(ids, new_token_count):
    """Every step reruns the whole sequence. Returns the tokens and each step's time."""
    tokens, times = [], []
    for _ in range(new_token_count):
        start = time.perf_counter()
        # generate logits based on all tokens seen so far input. No cache
        logits = model(input_ids=ids, use_cache=False).logits
        
        # last token vector gives you logits for next token 
        nxt = logits[:, -1].argmax(-1, keepdim=True)        
        times.append(time.perf_counter() - start)
        tokens.append(nxt.item())
        # append recently generated token into existing input
        ids = torch.cat([ids, nxt], dim=1)
    return tokens, times


def generate_with_cache(ids, new_token_count):
    """Prefill once, then feed only the newest token. Returns the tokens and each step's time."""
    tokens, times = [], []
    start = time.perf_counter()
    # prefill stage. use cache
    out = model(input_ids=ids, use_cache=True)
    for _ in range(new_token_count):
        # last token vector gives you logits for next token 
        nxt = out.logits[:, -1].argmax(-1, keepdim=True)
        times.append(time.perf_counter() - start)
        tokens.append(nxt.item())
        start = time.perf_counter()
        # only recently generated token is sent as input 
        out = model(input_ids=nxt, past_key_values=out.past_key_values, use_cache=True)
    return tokens, times


with torch.no_grad():
    # Same tokens either way: the cache stores values that never change
    prompt_tokens = tok("The hungry dog ate the bone because", return_tensors="pt")["input_ids"]
    a, _ = generate_without_cache(prompt_tokens, 20)
    b, _ = generate_with_cache(prompt_tokens, 20)
    print(f"identical tokens: {a == b}")
    print(f"generated: {tok.decode(b)!r}\n")

    # Time per generated token as the context grows. Step 1 is the prefill in both
    # methods, so it is left out; the median keeps one slow step from skewing a row.
    filler = tok("The hungry dog ate the bone. " * 200, return_tensors="pt")["input_ids"]
    steps = 6
    print(f"{'context':>8} {'no cache':>12} {'cache':>10} {'ratio':>7}")
    for n in (64, 256, 512, 1024):
        ids = filler[:, :n]
        _, t_none = generate_without_cache(ids, steps)
        _, t_cache = generate_with_cache(ids, steps)
        none_ms = 1000 * statistics.median(t_none[1:])
        cache_ms = 1000 * statistics.median(t_cache[1:])
        print(f"{n:>8} {none_ms:>9.0f} ms {cache_ms:>7.0f} ms {none_ms / cache_ms:>6.1f}×")

identical tokens: True
generated: ' it was a bone. The dog is a hungry dog. The dog is a hungry dog. The'

 context     no cache      cache   ratio
      64        93 ms      37 ms    2.5×
     256       242 ms      36 ms    6.8×
     512       406 ms      48 ms    8.5×
    1024      1378 ms      39 ms   35.3×


## 2. Shapes and bytes · §5.2

**Predict before you run:**

1. The shape of one layer's K tensor for a 256-token prompt. Read `num_hidden_layers`, `num_key_value_heads` and `head_dim` from the model's [`config.json`](https://huggingface.co/Qwen/Qwen3-0.6B/blob/main/config.json).
2. The total KV bytes, from $2 \times \text{layers} \times \text{kv\_heads} \times \text{head\_dim} \times \text{tokens} \times 2$ bytes.
3. The context length at which one conversation's cache outweighs the weights.

In [5]:
# Shapes and bytes, measured against the formula
# One prefill pass fills the cache for every prompt token.
inputs = tok("The hungry dog ate the bone. " * 37, return_tensors="pt")
inputs["input_ids"] = inputs["input_ids"][:, :256]
inputs["attention_mask"] = inputs["attention_mask"][:, :256]
with torch.no_grad():
    cache = model(**inputs, use_cache=True).past_key_values


def kv_per_layer(cache):
    """(K, V) per layer, across transformers' cache layouts."""
    if hasattr(cache, "layers"):        # DynamicCache, transformers >= 4.56
        return [(layer.keys, layer.values) for layer in cache.layers]
    if hasattr(cache, "key_cache"):     # DynamicCache, older releases
        return list(zip(cache.key_cache, cache.value_cache))
    return list(cache)                  # tuple of (K, V) pairs


kv = kv_per_layer(cache)
n = inputs["input_ids"].shape[1]
k0, v0 = kv[0]

print(f"tokens cached:      {n}")
print(f"layers:             {len(kv)}")
print(f"one layer's K:      {tuple(k0.shape)}   (batch, kv_heads, tokens, head_dim)")
print(f"one layer's V:      {tuple(v0.shape)}   dtype {k0.dtype}")

layer_k = k0.numel() * k0.element_size()
print(f"\none layer's K:      {layer_k:,} bytes   = {' × '.join(map(str, k0.shape))} × {k0.element_size()}")
print(f"K + V, one layer:   {2 * layer_k:,}")
print(f"× {len(kv)} layers:        {2 * layer_k * len(kv):,}")

measured = sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in kv)
formula = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * cfg.head_dim * n * 2

print(f"\nmeasured KV bytes:  {measured:,}   (summed over every cached tensor)")
print(f"formula (head_dim={cfg.head_dim}): {formula:,}   match: {measured == formula}")

per_token = measured / n
weights = sum(p.numel() * p.element_size() for p in model.parameters())
print(f"\nKV per token:       {per_token / 1024:.0f} KiB")
print(f"weights:            {weights / 1e9:.2f} GB")
print(f"KV = weights at:    {weights / per_token:,.0f} tokens")

tokens cached:      256
layers:             28
one layer's K:      (1, 8, 256, 128)   (batch, kv_heads, tokens, head_dim)
one layer's V:      (1, 8, 256, 128)   dtype torch.bfloat16

one layer's K:      524,288 bytes   = 1 × 8 × 256 × 128 × 2
K + V, one layer:   1,048,576
× 28 layers:        29,360,128

measured KV bytes:  29,360,128   (summed over every cached tensor)
formula (head_dim=128): 29,360,128   match: True

KV per token:       112 KiB
weights:            1.19 GB
KV = weights at:    10,394 tokens


## 3. Context: when one sequence outweighs the weights · §5.2

Every cached token adds the same 112 KiB, so the cache grows **linearly with the context window**. The weights do not grow at all.

**Predict before you run:** at what context length does a single sequence's cache outweigh the 1.19 GB of weights? Where does that move for a batch of 8?

In [ ]:
import pandas as pd

# Context scales the cache linearly: check it on a real 512-token prefill
long_ids = tok("The hungry dog ate the bone. " * 80, return_tensors="pt")["input_ids"][:, :512]
with torch.no_grad():
    long_cache = model(input_ids=long_ids, use_cache=True).past_key_values
long_bytes = sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in kv_per_layer(long_cache))
print(f"512 tokens: {long_bytes:,} bytes = {long_bytes / measured:.0f}× the {n}-token cache\n")

batches = (1, 8)
weights_gb = weights / 1e9
max_ctx = cfg.max_position_embeddings
rows = {}
for ctx in (512, 1024, 2048, 4096, 8192, 16384, 32768, max_ctx):
    row = {}
    for b in batches:
        kv_gb = b * ctx * per_token / 1e9
        mark = "  KV larger" if kv_gb > weights_gb else ""
        row[(f"batch {b}", "KV cache (GB)")] = round(kv_gb, 2)
        row[(f"batch {b}", "KV ÷ weights")] = f"{kv_gb / weights_gb:.2f}×{mark}"
    rows[f"{ctx:,}" + (" (max)" if ctx == max_ctx else "")] = row

print(f"weights: {weights_gb:.2f} GB")
for b in batches:
    print(f"batch {b}: the cache overtakes the weights at {weights / (b * per_token):,.0f} tokens per sequence")
table = pd.DataFrame.from_dict(rows, orient="index")
table.columns = pd.MultiIndex.from_tuples(table.columns)
table.index.name = "context"
table

## 4. Batch: when the cache outweighs the weights · §5.3

The weights are loaded once and shared by every request. The cache is **per sequence**: each request in a batch holds its own keys and values. So the weights stay fixed while the cache grows with **batch × context**.

**Predict before you run:** at 2,048 tokens per sequence, at which batch size does the cache outweigh Qwen3-0.6B's 1.19 GB of weights? And at 8,192?

In [ ]:
# Batch scales the cache linearly: check it on a real batched prefill
batch = 4
with torch.no_grad():
    batched = model(input_ids=inputs["input_ids"].repeat(batch, 1), use_cache=True).past_key_values
batched_bytes = sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in kv_per_layer(batched))
print(f"batch {batch} × {n} tokens: {batched_bytes:,} bytes = {batched_bytes / measured:.0f}× the batch-1 cache\n")

contexts = (2048, 8192)  # tokens per sequence
weights_gb = weights / 1e9
rows = {}
for b in (1, 2, 4, 8, 16, 32, 64):
    row = {}
    for ctx in contexts:
        kv_gb = b * ctx * per_token / 1e9
        mark = "  KV larger" if kv_gb > weights_gb else ""
        row[(f"{ctx:,} tokens", "KV cache (GB)")] = round(kv_gb, 2)
        row[(f"{ctx:,} tokens", "KV ÷ weights")] = f"{kv_gb / weights_gb:.2f}×{mark}"
    rows[b] = row

print(f"weights: {weights_gb:.2f} GB, shared by every sequence")
for ctx in contexts:
    print(f"at {ctx:,} tokens per sequence, the cache overtakes the weights at batch {weights / (ctx * per_token):.1f}")
table = pd.DataFrame.from_dict(rows, orient="index")
table.columns = pd.MultiIndex.from_tuples(table.columns)
table.index.name = "batch"
table